Referee data is unique in that it can't be retrieved from an NHL API endpoint and has to be scraped from HTML post-game summaries. As a result, there is a risk of data errors. This notebook explores and cleans raw referee data and prepares two tables:
1. The normalized referee table for use in the final SQL schema.
2. A temporary table containing game_id and referee_id columns for mapping penalties (which are associated with game_ids) to the referee(s) that called them.

In [107]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)

p = Path('../../Data/Raw/referees_raw.csv')
if not p.exists():
    print('Raw referee data must be extracted first via the Silver/Scripts/Build Tables/referees.py script.')
    sys.exit(1)

df_referees = pd.read_csv(p)
df_referees.drop(columns=['linesman_1', 'linesman_2'], inplace=True)
df_referees.head()

,game_id,referee_1,referee_2
0,2021020001,#11 Kelly Sutherland,#40 Steve Kozari
1,2021020002,#24 Graham Skilliter,#8 Francois StLaurent
2,2021020003,#7 Garrett Rank,#19 Gord Dwyer
3,2021020004,#5 Chris Rooney,#4 Wes McCauley
4,2021020005,#24 Graham Skilliter,#3 Chris Schlenker


I was previously aware that some games were officiated using a three-man system (one referee and two linesmen). Let's see how many.

In [108]:
df_one_ref = df_referees[(df_referees['referee_1'].isna())|(df_referees['referee_1']=='')|(df_referees['referee_2'].isna())|(df_referees['referee_2']=='')]
df_one_ref

,game_id,referee_1,referee_2
464,2021020465,#37 Pierre Lambert,NaN
467,2021020468,#9 Dan O'Rourke,NaN
1854,2022020543,#10 Kyle Rehman,NaN
6318,2025021071,#34 Brandon Schrader,NaN


I did some research and discovered the reason for each absence.
* 2021020465: Dec 17, 2021 VGK vs NYR game. Kendrick Nicholson was scheduled to officiate but was diagnosed with COVID-19 [[1](https://scoutingtherefs.com/2021/12/32893/report-up-to-13-nhl-officials-in-covid-protocol/)].
* 2021020468: Dec 17, 2021 DAL vs STL game. Mitch Dunning was scheduled to officiate but was diagnosed with COVID-19 [[1](https://scoutingtherefs.com/2021/12/32893/report-up-to-13-nhl-officials-in-covid-protocol/)].
* 2022020543: Dec 23, 2022 LAK vs ARI game. Tom Chmielewski was scheduled to officiate but could not reach the arena due to severe weather [[2](https://apnews.com/article/nhl-sports-arizona-coyotes-los-angeles-kings-bc196c54c07470a3a1dfdd80245f8994)].
* 2025021071: Mar 17, 2026 NSH vs WPG game. Chris Lee was scheduled to officiate but could not reach the arena due to severe weather [[3](https://www.rmoutlook.com/national-sports/jets-forced-lone-referee-to-work-overtime-lose-4-3-in-shootout-to-predators-12021141)].

Next, let's determine if any referees share a common name, as is the case with players. Let's also check for misspellings while we're at it, which would present as a similar value with a low count.

In [109]:
df_referees = pd.concat([df_referees[['game_id','referee_1']].rename(columns={'referee_1' : 'referee'}), df_referees[['game_id','referee_2']].rename(columns={'referee_2' : 'referee'})])
df_referees.dropna(inplace=True)
df_referees['referee'].value_counts()

referee
#4 Wes McCauley              375
#14 Trevor Hanson            374
#10 Kyle Rehman              374
#30 Kendrick Nicholson       373
#21 TJ Luxmore               373
#24 Graham Skilliter         372
#7 Garrett Rank              372
#23 Corey Syvret             371
#13 Furman South             371
#11 Kelly Sutherland         366
#26 Jake Brenk               365
#17 Frederick L'Ecuyer       363
#38 Peter MacDougall         361
#18 Tom Chmielewski          360
#31 Michael Markovic         352
#27 Eric Furlatt             350
#2 Jon McIsaac               346
#39 Brandon Blandina         345
#28 Chris Lee                341
#5 Chris Rooney              329
#19 Gord Dwyer               326
#16 Brian Pochmara           322
#3 Chris Schlenker           317
#22 Ghislain Hebert          304
#8 Francois StLaurent        302
#15 Jean Hebert              300
#9 Dan O'Rourke              295
#6 Francis Charron           293
#20 Mitch Dunning            273
#40 Steve Kozari             235
#3

This led to a funny insight: **when a referee wears #35 it means their luggage was lost in transit to the game!** The only other low count value was Ben Betker, a minor league referee who had a chance to officiate an NHL game, so not erroneous.

There are some repeated names though, differentiated by sweater number. To ensure they are the same person, I search for each at https://nhlofficials.com/officials. Conclusion: there are no referees with common names.

There's one small issue: the surnames St. Laurent and St. Pierre lost their periods and spaces. Just need to fix that and then package up the two tables.

In [110]:
df_referees['referee'] = df_referees.apply(lambda row: ' '.join(row['referee'].split(' ')[1:]), axis=1)
df_games_refs = df_referees.copy()

unique_referees = df_referees['referee'].unique()
df_referees_normalized = pd.DataFrame(unique_referees, columns=['referee'])
df_referees_normalized.reset_index(inplace=True)
df_referees_normalized.rename(columns={'index' : 'id'}, inplace=True)
df_referees_normalized['id'] = df_referees_normalized.apply(lambda row: row['id']+1, axis=1)

df_games_refs['referee_id'] = df_games_refs.apply(lambda row: df_referees_normalized[df_referees_normalized['referee']==row['referee']].iloc[-1, 0], axis=1)
df_games_refs.drop(columns=['referee'], inplace=True)
df_games_refs.sort_values(by='game_id', inplace=True)
print(df_games_refs.head())

df_referees_normalized['first'] = df_referees_normalized.apply(lambda row: row['referee'].split(' ')[0], axis=1)
df_referees_normalized['last'] = df_referees_normalized.apply(lambda row: row['referee'].split(' ')[1], axis=1)
df_referees_normalized.loc[df_referees_normalized['last']=='StLaurent', 'last'] = 'St. Laurent'
df_referees_normalized.loc[df_referees_normalized['last']=='StPierre', 'last'] = 'St. Pierre'
df_referees_normalized.drop(columns=['referee'], inplace=True)
df_referees_normalized.head()


      game_id  referee_id
0  2021020001           1
0  2021020001          22
1  2021020002          13
1  2021020002           2
2  2021020003           8


,id,first,last
0,1,Kelly,Sutherland
1,2,Graham,Skilliter
2,3,Garrett,Rank
3,4,Chris,Rooney
4,5,Kyle,Rehman


In [111]:
df_games_refs.to_csv('../../Data/Raw/game_ref_mapping.csv', index=False)
df_referees_normalized.to_csv('../../Data/Tables/referees.csv', index=False)